# Week 3a.1: Short-term Memory

Our agents so far have amnesia. Every `invoke` starts from nothing: the model does not remember what you asked in the previous message. 

Today we will address this problem for a single conversation.
- This doesn't apply to memory across conversations and sessions

In [1]:
from dotenv import load_dotenv
import os
import logging

load_dotenv()
assert os.getenv("GOOGLE_API_KEY"), "No GOOGLE_API_KEY found."

# silence a noisy advisory warning from the Google SDK
logging.getLogger("google_genai.models").setLevel(logging.ERROR)

print("API key loaded")

API key loaded


## 0. Model & Tool Setup

Execute one of the following to define the model.

If using Ollama, you will need to start it first (simply open the chat UI and send a message):
- https://docs.langchain.com/oss/python/integrations/chat/ollama

Verify that Ollama is serving a model:
- http://localhost:11434/

In [2]:
from langchain_ollama import ChatOllama

model = ChatOllama(model="qwen3.5:4b", reasoning=False)

In [ ]:
from langchain.chat_models import init_chat_model

model = init_chat_model(model="gpt-4.1-mini")

In [ ]:
from langchain_google_genai import ChatGoogleGenerativeAI

model = ChatGoogleGenerativeAI(model="gemini-3.5-flash-lite")

In [3]:
import requests
from langchain.tools import tool
from langchain_tavily import TavilySearch
from typing import Dict, Any
from datetime import datetime

# WEB SEARCH
@tool
def search_the_web(query: str) -> Dict[str, Any]:
    """Search the web for information"""
    tavily = TavilySearch(max_results=3)
    query_dict = {"query": query}
    results = tavily.invoke(query_dict)
    return results

# CURRENT TIME
@tool
def get_current_time() -> str:
    """Return the current local date and time."""
    return datetime.now().strftime("%A, %B %d, %Y at %I:%M %p")

# WEATHER
@tool
def get_weather(city: str) -> str:
    """Get the weather data for the city provided as an argument"""

    data = requests.get(f"https://wttr.in/{city}?format=j1").json()
    return data["current_condition"][0]

## 1. The problem

In [4]:
response = model.invoke("Hello from Boston.")
print(response.text)

Hello! It's great to meet you in the city of trees and cranes. How is your day going? Is there anything I can help you with regarding weather, local recommendations, or just a nice chat?


In [5]:
response = model.invoke("Where am I located?")
print(response.text)

I don't have access to information about your specific location unless you provide it to me. To find out where you are, you can:

1. Check your device settings (e.g., Google Maps on Android/iOS)
2. Use location-based apps or services like Siri, Google Assistant
3. Look for local signs, landmarks, or time zone references
4. Ask a local person for directions

Would you like help finding something specific near your location?


The model has no idea. It is **stateless**: nothing carries over from one call to the next. What looked like a conversation in a chat app was never memory inside the model; the application was resending the history every time.

We already have the tool for this: a call can take a **list of messages**. That list is the memory.

That is all short-term memory is: the application rereads the entire conversation to the model on every single call. Nothing is stored inside the model.

## 2. Agents with Threads

For agents, LangChain provides a built-in method to save previous messages into the agent's state (or memory).
- https://docs.langchain.com/oss/python/langchain/short-term-memory

We need to add a **checkpointer** (InMemorySaver) when invoking the `create_agent` method, which will save the conversation state after every call, filed under a `thread_id` we choose. The `thread_id` is typically the unique conversation ID.

Messages with the same thread are saved to the same memory

Add this to agent **declaration**: 
- `checkpointer = InMemorySaver()`

The thread_id is passed as a configurable to agent **invocation**:
- `config = {"configurable": {"thread_id": "1"}}`


In [6]:
from langchain.agents import create_agent
from langgraph.checkpoint.memory import InMemorySaver

agent = create_agent(
    model=model,
    tools=[get_weather, search_the_web, get_current_time],
    system_prompt="You're a helpful assistant who answers users' questions concisely.",
    #TODO: add the checkpointer
    checkpointer=InMemorySaver(),
)


In [7]:
#TODO define the config
config = {"configurable": {"thread_id": "1"}}

# add this after the messages dictionary in your agent invocation

In [8]:
from langchain.messages import HumanMessage

message = [HumanMessage(content="Hello from Boston!")]

result = agent.invoke({"messages": message}, config)
print(result["messages"][-1].text)

Hi there! It's nice to meet you from Boston. How are things over there?


In [9]:
message2 = [HumanMessage(content="Where am I located?")]
result = agent.invoke({"messages": message2}, config)
print(result["messages"][-1].text)

You are located in **Boston**. Did you mean to ask for the weather here?


In [10]:
result

{'messages': [HumanMessage(content='Hello from Boston!', additional_kwargs={}, response_metadata={}, id='55e82bc3-0d5f-4de4-80b0-0350652427e5'),
  AIMessage(content="Hi there! It's nice to meet you from Boston. How are things over there?", additional_kwargs={}, response_metadata={'model': 'qwen3.5:4b', 'created_at': '2026-09-21T23:26:54.925301Z', 'done': True, 'done_reason': 'stop', 'total_duration': 1970981292, 'load_duration': 2028833, 'prompt_eval_count': 394, 'prompt_eval_duration': 1249415000, 'eval_count': 19, 'eval_duration': 693719000, 'logprobs': None, 'model_name': 'qwen3.5:4b', 'model_provider': 'ollama'}, id='lc_run--01a0c64b-3619-7230-906a-4c540b709743-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 394, 'output_tokens': 19, 'total_tokens': 413}),
  HumanMessage(content='Where am I located?', additional_kwargs={}, response_metadata={}, id='12b10d8a-4a1b-49a9-8410-09d5d276bc5e'),
  AIMessage(content='You are located in **Boston**. Did you mean to a

In [11]:
message3 = [HumanMessage(content="What's the weather like?")]

result = agent.invoke({"messages": message3}, config)
print(result["messages"][-1].text)

The weather in Boston is currently **Sunny** with a temperature of **60°F**. The wind speed is low at 6 mph, and there's very little cloud cover. It looks like a comfortable evening!


In [12]:
result

{'messages': [HumanMessage(content='Hello from Boston!', additional_kwargs={}, response_metadata={}, id='55e82bc3-0d5f-4de4-80b0-0350652427e5'),
  AIMessage(content="Hi there! It's nice to meet you from Boston. How are things over there?", additional_kwargs={}, response_metadata={'model': 'qwen3.5:4b', 'created_at': '2026-09-21T23:26:54.925301Z', 'done': True, 'done_reason': 'stop', 'total_duration': 1970981292, 'load_duration': 2028833, 'prompt_eval_count': 394, 'prompt_eval_duration': 1249415000, 'eval_count': 19, 'eval_duration': 693719000, 'logprobs': None, 'model_name': 'qwen3.5:4b', 'model_provider': 'ollama'}, id='lc_run--01a0c64b-3619-7230-906a-4c540b709743-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 394, 'output_tokens': 19, 'total_tokens': 413}),
  HumanMessage(content='Where am I located?', additional_kwargs={}, response_metadata={}, id='12b10d8a-4a1b-49a9-8410-09d5d276bc5e'),
  AIMessage(content='You are located in **Boston**. Did you mean to a

Let's now pass in a different thread:

In [13]:
config = {"configurable": {"thread_id": "2"}}
result = agent.invoke({"messages": message2}, config)
print(result["messages"][-1].text)

I don't have access to your specific location. To help you find the weather for your area, please tell me your city name.


## 3. A chat loop

With threads, a real chat interface is a few lines. Uncomment and run; type `quit` to stop.

In [14]:
while True:
    user = input("You: ")

    if user.lower() in {"quit", "exit"}:
        break

    message = [HumanMessage(content=user)]

    result = agent.invoke({"messages": message}, config)
    
    print("Assistant:", result["messages"][-1].text)

Assistant: It's overcast and feels like 66°F (22°C) in New York City today. The air is a bit humid at 41%, with a light wind coming from the ENE direction at 10 mph. No precipitation expected right now.
Assistant: There are several movies playing in New York City on September 22nd (tomorrow), including:

**New Releases & Features:**
*   **Ice Cream Man** (NR)
*   **The Storm**
*   **Stranglehold**
*   **Death of a Yellow Bird**
*   **Orfeo**
*   **The Uprising**
*   **Don’t Look Back In Anger**

**Currently Showing:**
*   **Star Trek II: The Wrath of Khan** (Director's Cut)
*   **A Clockwork Orange**
*   **Akira** (4K)
*   **Onslaught**
*   **The Dog Stars**
*   **Fall 2: Deadpoint**

For the most current showtimes and ticketing, checking a site like [NYC.com](https://www.nyc.com/movies) or an aggregator would be helpful as schedules are constantly updated.


## 4. A chat window

The terminal loop above works, but one import gives us a real chat interface. `agentui.py` lives next to this notebook: it wraps any agent built with `create_agent`, shows tool calls as collapsible entries while the agent works, and sends the same `thread_id` on every turn, so the agent's own checkpointer does the remembering.

Run the cell and open the local URL it prints. Interrupt the kernel (or restart it) to stop the server.

In [15]:
from agentui import GradioUI

app = GradioUI(agent, {"configurable": {"thread_id": "gradio-demo"}}, title="Agent with Memory")
app.launch()

* Running on local URL:  http://127.0.0.1:7861
* To create a public link, set `share=True` in `launch()`.


## 5. Tracing

[LangSmith](https://smith.langchain.com/) is LangChain's official tool for tracing agent calls and executions:

To set it up, get an API key from https://smith.langchain.com/ and add it your .env file.
- Set `LANGSMITH_TRACING=true`